In [2]:
import pandas as pd

df_clean = pd.read_csv("../data/fraudTrain_featured.csv")

df_clean.shape

(1296675, 30)

In [3]:
df_clean[
    ["transaction_hour", "amount_band", "category", "state", "is_fraud"]
].head()

,transaction_hour,amount_band,category,state,is_fraud
0,0,0-50,misc_net,NC,0
1,0,100-500,grocery_pos,WA,0
2,0,100-500,entertainment,ID,0
3,0,0-50,gas_transport,MT,0
4,0,0-50,misc_pos,VA,0


In [4]:
def fraud_alert(row):
    # Critical Rule 1
    if(row["transaction_hour"]  in [22,23]
       and row["amount_band"] == "500-1k"
       and row["category"] == "misc_net"):
       return "Critical"

     # Critical Rule 2
    elif(row["transaction_hour"] in [22,23]
        and row["amount_band"] == "1k-5k"
        and row["category"] == "shopping_net"):
        return "Critical"

    # High Rule 1
    elif(row["transaction_hour"] in [22,23]
         and row["amount_band"] == "500-1k"
         and row["category"] == "shopping_net"):
        return "High"
    
    # High Rule 2
    elif(row["transaction_hour"] in [22,23]
         and row["amount_band"] == "100-500"
         and row["category"] == "grocery_pos"):
         return "High"

    #Medium Rule
    elif(row["transaction_hour"] in [22,23]
         and row["amount_band"] in ["500-1k","1k-5k"]):
        return "Medium"

    else:
        return "Low"
    

In [5]:
df_clean["risk_level"] = df_clean.apply(fraud_alert,axis = 1)


In [14]:
df_clean["risk_level"].value_counts()

risk_level
Low         1292466
Medium         1719
Critical       1319
High           1171
Name: count, dtype: int64

In [16]:
risk_analysis = df_clean.groupby("risk_level").agg(total_transaction = ("is_fraud","count"),
                                             fraud_transaction = ("is_fraud","sum"))

In [17]:
risk_analysis["fraud_rate"] = (risk_analysis["fraud_transaction"]/risk_analysis["total_transaction"])*100

In [18]:
risk_analysis.sort_values("fraud_rate",ascending = False)

,total_transaction,fraud_transaction,fraud_rate
risk_level,,,
Critical,1319,1202,91.129644
High,1171,942,80.444065
Medium,1719,650,37.812682
Low,1292466,4712,0.364574


In [19]:
alert_summary = (df_clean["risk_level"].value_counts().rename_axis("risk_level").reset_index(name = "transaction"))

In [20]:
alert_summary["alert_percentage"] = (alert_summary["transaction"]/len(df_clean))*100

alert_summary

,risk_level,transaction,alert_percentage
0,Low,1292466,99.675401
1,Medium,1719,0.132570
2,Critical,1319,0.101722
3,High,1171,0.090308


In [21]:
risk_analysis = df_clean.groupby("risk_level").agg(total_transaction = ("is_fraud","count"),
                                             fraud_transaction = ("is_fraud","sum"))

risk_analysis["fraud_rate"] = (risk_analysis["fraud_transaction"]/risk_analysis["total_transaction"])*100

risk_analysis["fraud_capture_rate"] = (risk_analysis["fraud_transaction"]/df_clean["is_fraud"].sum())*100

risk_analysis.sort_values("fraud_rate",ascending = False)

,total_transaction,fraud_transaction,fraud_rate,fraud_capture_rate
risk_level,,,,
Critical,1319,1202,91.129644,16.013856
High,1171,942,80.444065,12.549960
Medium,1719,650,37.812682,8.659739
Low,1292466,4712,0.364574,62.776446


In [47]:
alert_transaction = df_clean[df_clean["risk_level"].isin(["Medium","High","Critical"])]

In [48]:
alert_total = len(alert_transaction)

In [49]:
alert_fraud = alert_transaction["is_fraud"].sum()

In [50]:
alert_fraud_rate = (alert_fraud/alert_total)*100

In [52]:
overall_fraud = df_clean["is_fraud"].sum()

In [53]:
fraud_capture_rate = (alert_fraud/overall_fraud)*100

In [54]:
print("Alerted Transactions:", alert_total)
print("Fraud in Alerts:", alert_fraud)
print("Alert Fraud Rate:", round(alert_fraud_rate, 2), "%")
print("Fraud Capture Rate:", round(fraud_capture_rate, 2), "%")

Alerted Transactions: 4209
Fraud in Alerts: 2794
Alert Fraud Rate: 66.38 %
Fraud Capture Rate: 37.22 %


In [7]:
rule_summary = pd.DataFrame({

    "Rule" : ["R1","R2","R3","R4","R5"],

    "Condition" : [
        "22-23 + 500-1k + misc_net",
        "22-23 + 1k-5k + Shooping_net",
        "22-23 + 500-1k + Shopping_net",
        "22-23 + 100-500 + grocery_pos",
        "22-23 + 500-5k"
    ],

    "Alert Level" : [
        "Critical",
        "critical",
        "High",
        "High",
        "Medium"
    ],
    "Historical Evidence" : [
         "94–97%",
        "85–89%",
        "~77%",
        "~89–93%",
        "Calculate"
    ]    
})

rule_summary

,Rule,Condition,Alert Level,Historical Evidence
0,R1,22-23 + 500-1k + misc_net,Critical,94–97%
1,R2,22-23 + 1k-5k + Shooping_net,critical,85–89%
2,R3,22-23 + 500-1k + Shopping_net,High,~77%
3,R4,22-23 + 100-500 + grocery_pos,High,~89–93%
4,R5,22-23 + 500-5k,Medium,Calculate


POWER_BI

In [77]:
df_powerbi = df_clean.copy()

df_powerbi["risk_level"] = df_powerbi["risk_level"].astype(str)

In [78]:
df_powerbi["fraud_status"] = df_powerbi["is_fraud"].map({
    0: "Legitimate",
    1: "Fraud"
})

In [79]:
alert_flag = {
    "Low": "No Alert",
    "Medium": "Alert",
    "High": "Alert",
    "Critical": "Alert"
}
df_powerbi["alert_flag"] = df_powerbi["risk_level"].map(alert_flag)

In [80]:
risk_score = {
    "Low":0,
    "Medium":1,
    "High":2,
    "Critical":3
}
df_powerbi["risk_score"] = df_powerbi["risk_level"].map(risk_score)

In [97]:
  df_powerbi[
        ["is_fraud", "fraud_status", "risk_level",
         "alert_flag", "risk_score"]
    ].sample(5)

,is_fraud,fraud_status,risk_level,alert_flag,risk_score
817232,0,Legitimate,Low,No Alert,0
502551,0,Legitimate,Low,No Alert,0
922561,0,Legitimate,Low,No Alert,0
1092306,0,Legitimate,Low,No Alert,0
617376,0,Legitimate,Low,No Alert,0


In [98]:
df_powerbi.to_csv(
    "../data/fraud_analysis_final.csv",
    index=False
)

print("Final Power BI dataset saved successfully!")

Final Power BI dataset saved successfully!
